# Cascade inference — full corpus

Runs all 4 cascade models on 737k chunks and outputs `chunk_commitment_cascade.parquet`.

In [ ]:
!pip install -q -U transformers datasets scikit-learn

In [ ]:
import glob, json
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification

DEVICE    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_LEN   = 128
BATCH     = 128
OUT_DIR   = Path("/kaggle/working")
print(f"Device: {DEVICE}")


In [ ]:
def find_input_file(*names):
    for name in names:
        for pattern in [f"/kaggle/input/**/{name}", f"/kaggle/input/datasets/kevinnchan/**/{name}"]:
            matches = glob.glob(pattern, recursive=True)
            if matches:
                return Path(matches[0])
        local = Path(name)
        if local.exists(): return local
    raise FileNotFoundError(f"Cannot find any of {names}")


In [ ]:
def find_input_dir(*names):
    for name in names:
        for pattern in [f"/kaggle/input/**/{name}", f"/kaggle/input/datasets/kevinnchan/**/{name}"]:
            matches = glob.glob(pattern, recursive=True)
            dirs = [m for m in matches if Path(m).is_dir()]
            if dirs: return Path(dirs[0])
    raise FileNotFoundError(f"Cannot find dir: {names}")


In [ ]:
# ── Load all 4 models (find by axis name in id2label.json, slug-agnostic) ─
def load_model(axis_name):
    candidates = glob.glob("/kaggle/input/**/id2label.json", recursive=True)
    for c in candidates:
        try:
            with open(c) as f: meta = json.load(f)
            if meta.get("axis") != axis_name: continue
            model_dir = Path(c).parent
            id2label  = {int(k): v for k, v in meta["id2label"].items()}
            label2id  = {v: k for k, v in id2label.items()}
            tok   = AutoTokenizer.from_pretrained(str(model_dir))
            model = AutoModelForSequenceClassification.from_pretrained(str(model_dir))
            model.eval().to(DEVICE)
            print(f"Loaded axis={axis_name} from {model_dir}  labels={id2label}")
            return tok, model, id2label, label2id
        except Exception:
            continue
    raise FileNotFoundError(f"No model found for axis: {axis_name}")

tok1a, model1a, id2label1a, label2id1a = load_model("buyin_relevance")
tok2a, model2a, id2label2a, label2id2a = load_model("buyin_direction")
tok1b, model1b, id2label1b, label2id1b = load_model("stance_relevance")
tok2b, model2b, id2label2b, label2id2b = load_model("stance_direction")

In [ ]:
# ── Load chunk text ───────────────────────────────────────────────────────
def load_chunks():
    cc_path = find_input_file("comments_chunks.parquet")
    sc_path = find_input_file("submissions_chunks.parquet")
    cc = pd.read_parquet(cc_path, columns=["chunk_id", "doc_id", "text"])
    sc = pd.read_parquet(sc_path, columns=["chunk_id", "doc_id", "text"])
    df = pd.concat([cc, sc], ignore_index=True).drop_duplicates("chunk_id")
    df["text"] = df["text"].fillna("").astype(str).str.strip()
    df = df[df["text"].str.len() > 5].reset_index(drop=True)
    print(f"Chunks loaded: {len(df):,}")
    return df

chunks = load_chunks()
texts = chunks["text"].tolist()
chunk_ids = chunks["chunk_id"].tolist()


In [ ]:
# ── Inference helper ──────────────────────────────────────────────────────
class TextDataset(Dataset):
    def __init__(self, texts, tokenizer):
        self.texts = texts
        self.tok   = tokenizer

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tok(self.texts[idx], truncation=True, max_length=MAX_LEN,
                       padding="max_length", return_tensors="pt")
        return {k: v.squeeze(0) for k, v in enc.items()}


@torch.no_grad()
def run_inference(texts, tokenizer, model, batch_size=BATCH, desc=""):
    ds     = TextDataset(texts, tokenizer)
    loader = DataLoader(ds, batch_size=batch_size, num_workers=2, pin_memory=True)
    all_probs = []
    for i, batch in enumerate(loader):
        batch  = {k: v.to(DEVICE) for k, v in batch.items()}
        logits = model(**batch).logits
        probs  = torch.softmax(logits, dim=-1).cpu().numpy()
        all_probs.append(probs)
        if i % 50 == 0:
            print(f"  {desc} {i*batch_size:,}/{len(texts):,}", end="\r")
    print(f"  {desc} done — {len(texts):,} chunks")
    return np.concatenate(all_probs, axis=0)


In [ ]:
# ── Stage 1a: buyin relevance ─────────────────────────────────────────────
print("Stage 1a: buyin relevance ...")
probs1a  = run_inference(texts, tok1a, model1a, desc="1a")
# id2label1a: {0: 'not_relevant', 1: 'relevant'} (or whichever order the model used)
rel_col  = label2id1a.get("relevant", label2id1a.get(1, 1))
not_col  = label2id1a.get("not_relevant", label2id1a.get(0, 0))
is_relevant = probs1a[:, rel_col] >= 0.5
print(f"  Relevant: {is_relevant.sum():,} / {len(is_relevant):,} ({is_relevant.mean():.1%})")


In [ ]:
# ── Stage 2a: committed vs uncommitted (relevant chunks only) ─────────────
print("Stage 2a: committed vs uncommitted ...")
rel_texts    = [texts[i] for i in range(len(texts)) if is_relevant[i]]
rel_indices  = [i for i in range(len(texts)) if is_relevant[i]]
probs2a      = run_inference(rel_texts, tok2a, model2a, desc="2a")
com_col      = label2id2a.get("committed",   0)
unc_col      = label2id2a.get("uncommitted", 1)

# Build full-corpus buyin arrays (default: neutral)
prob_buyin_committed   = np.zeros(len(texts))
prob_buyin_uncommitted = np.zeros(len(texts))
prob_buyin_neutral     = np.ones(len(texts))

for arr_idx, corpus_idx in enumerate(rel_indices):
    prob_buyin_committed[corpus_idx]   = probs2a[arr_idx, com_col]
    prob_buyin_uncommitted[corpus_idx] = probs2a[arr_idx, unc_col]
    prob_buyin_neutral[corpus_idx]     = 0.0

# Non-relevant chunks: set neutral prob from stage 1a
for i in range(len(texts)):
    if not is_relevant[i]:
        prob_buyin_neutral[i]     = probs1a[i, not_col]
        prob_buyin_committed[i]   = probs1a[i, rel_col] * 0.5
        prob_buyin_uncommitted[i] = probs1a[i, rel_col] * 0.5

buyin_label = []
for i in range(len(texts)):
    if not is_relevant[i]:
        buyin_label.append("neutral")
    else:
        if probs2a[rel_indices.index(i) if i in rel_indices else 0, com_col] >=            probs2a[rel_indices.index(i) if i in rel_indices else 0, unc_col]:
            buyin_label.append("committed")
        else:
            buyin_label.append("uncommitted")

# Rebuild properly
buyin_label = ["neutral"] * len(texts)
for arr_idx, corpus_idx in enumerate(rel_indices):
    if probs2a[arr_idx, com_col] >= probs2a[arr_idx, unc_col]:
        buyin_label[corpus_idx] = "committed"
    else:
        buyin_label[corpus_idx] = "uncommitted"

from collections import Counter
print(f"  Buyin label dist: {Counter(buyin_label)}")


In [ ]:
# ── Stage 1b: stance relevance ────────────────────────────────────────────
print("Stage 1b: stance relevance ...")
probs1b     = run_inference(texts, tok1b, model1b, desc="1b")
has_col     = label2id1b.get("has_stance", label2id1b.get(1, 1))
no_col      = label2id1b.get("no_stance",  label2id1b.get(0, 0))
has_stance  = probs1b[:, has_col] >= 0.5
print(f"  Has stance: {has_stance.sum():,} ({has_stance.mean():.1%})")


In [ ]:
# ── Stage 2b: supportive vs critical ──────────────────────────────────────
print("Stage 2b: supportive vs critical ...")
st_texts   = [texts[i] for i in range(len(texts)) if has_stance[i]]
st_indices = [i for i in range(len(texts)) if has_stance[i]]
probs2b    = run_inference(st_texts, tok2b, model2b, desc="2b")
sup_col    = label2id2b.get("supportive", 0)
crit_col   = label2id2b.get("critical",   1)

prob_stance_supportive = np.zeros(len(texts))
prob_stance_critical   = np.zeros(len(texts))
prob_stance_neutral    = np.ones(len(texts))

for arr_idx, corpus_idx in enumerate(st_indices):
    prob_stance_supportive[corpus_idx] = probs2b[arr_idx, sup_col]
    prob_stance_critical[corpus_idx]   = probs2b[arr_idx, crit_col]
    prob_stance_neutral[corpus_idx]    = 0.0

for i in range(len(texts)):
    if not has_stance[i]:
        prob_stance_neutral[i]    = probs1b[i, no_col]
        prob_stance_supportive[i] = probs1b[i, has_col] * 0.5
        prob_stance_critical[i]   = probs1b[i, has_col] * 0.5

stance_label = ["neutral"] * len(texts)
for arr_idx, corpus_idx in enumerate(st_indices):
    if probs2b[arr_idx, sup_col] >= probs2b[arr_idx, crit_col]:
        stance_label[corpus_idx] = "supportive"
    else:
        stance_label[corpus_idx] = "critical"

print(f"  Stance label dist: {Counter(stance_label)}")


In [ ]:
# ── Assemble output parquet ───────────────────────────────────────────────
result = pd.DataFrame({
    "chunk_id":               chunk_ids,
    "buyin_label":            buyin_label,
    "stance_label":           stance_label,
    "prob_buyin_committed":   prob_buyin_committed.round(5),
    "prob_buyin_uncommitted": prob_buyin_uncommitted.round(5),
    "prob_buyin_neutral":     prob_buyin_neutral.round(5),
    "prob_stance_supportive": prob_stance_supportive.round(5),
    "prob_stance_critical":   prob_stance_critical.round(5),
    "prob_stance_neutral":    prob_stance_neutral.round(5),
})

out_path = OUT_DIR / "chunk_commitment_cascade.parquet"
result.to_parquet(out_path, index=False)
print(f"\nSaved → {out_path}  ({result.shape})")
print(f"Buyin:  {result['buyin_label'].value_counts().to_dict()}")
print(f"Stance: {result['stance_label'].value_counts().to_dict()}")


In [ ]:
# ── Quick eval against testset ────────────────────────────────────────────
ts_path = find_input_file("commitment_testset.parquet")
ts = pd.read_parquet(ts_path)
merged = ts.merge(result, on="chunk_id", how="inner")
print(f"Testset rows with predictions: {len(merged)}")

from sklearn.metrics import classification_report
for gold_col, pred_col, name in [
    ("human_label",  "buyin_label",  "BUYIN"),
    ("human_stance", "stance_label", "STANCE"),
]:
    sub = merged[merged[gold_col].isin(["committed","uncommitted","neutral",
                                         "supportive","critical"])]
    print(f"\n{'='*60}")
    print(f"  CASCADE {name} — n={len(sub)}")
    print(classification_report(sub[gold_col], sub[pred_col], digits=3))
